<a href="https://colab.research.google.com/github/strangeworks/examples/blob/master/examples/optimization/gurobi/gurobi.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" /></a>

# Gurobi Optimization

This notebook demonstrates how to use Gurobi through the Strangeworks Optimization Service. Gurobi is an industry-leading solver for mathematical optimization that supports QUBO, MPS, and LP problem formats.


# Installation

In [ ]:
%pip install -q -U pip numpy strangeworks-optimization

## Authentication

Your API token can be found in the [portal](https://portal.strangeworks.com).

### Google Colab

If running in Google Colab first set your API token as a secret environment variable in Colab. You can do this by clicking on the key icon on the left, then adding a key called `STRANGEWORKS_API_KEY` with your API token as the value.

Then, run the cell below to authenticate.

In [ ]:
import strangeworks as sw
from google.colab import userdata
import warnings
warnings.filterwarnings('ignore', category=ImportWarning)

api_key = userdata.get('STRANGEWORKS_API_KEY')
sw.authenticate(api_key)

### Local

If running locally, you can save your API token in a dotenv file. Create a file called `.env` in the same directory as this notebook and add the following line to the file:

```
STRANGEWORKS_API_KEY=your_api_token
```

Then, run the cell below to authenticate.

In [ ]:
# %pip install -q python-dotenv
# import strangeworks as sw
# import os
# from dotenv import load_dotenv

# load_dotenv()
# api_key = os.getenv("STRANGEWORKS_API_KEY")
# sw.authenticate(api_key)

# StrangeworksOptimizer

We implement our own Optimization class called `StrangeworksOptimizer`.
`StrangeworksOptimizer` provides the full functionality of the Strangeworks Optimization Service.

In [2]:
from strangeworks_optimization import StrangeworksOptimizer
so = StrangeworksOptimizer()

## Models

Gurobi accepts the following models:

- `BinaryQuadraticModel` - [`dimod`](https://docs.ocean.dwavesys.com/en/stable/docs_dimod/) provides a shared API for samplers and we default to using `BinaryQuadraticModel` to store and represent QUBO problems. Use with `gurobi.qubo` solver.
- `MPSFile` - Mathematical Programming System format, a standard format for linear and mixed-integer programming problems. Use with `gurobi.mps` solver. **Note:** Gurobi supports MIQCP (Mixed-Integer Quadratically Constrained Programming) problems in MPS format, including non-convex MIQCP problems. See the [Gurobi MIQCP documentation](https://www.gurobi.com/faqs/gurobi-miqcp/) for more details.
- `LpFile` - Linear Programming format. Use with `gurobi.lp` solver.

Here is a simple QUBO problem defined by linear and quadratic terms. The linear terms are a dictionary where the keys are the variables and the values are the linear coefficients. The quadratic terms are a dictionary where the keys are the pairs of variables and the values are the quadratic coefficients. The BinaryQuadraticModel class is used to create the QUBO problem.

In [3]:
from dimod import BinaryQuadraticModel

linear = {1: -2, 2: -2, 3: -3, 4: -3, 5: -2}
quadratic = {(1, 2): 2, (1, 3): 2, (2, 4): 2, (3, 4): 2, (3, 5): 2, (4, 5): 2}
model = BinaryQuadraticModel(linear, quadratic, "BINARY")

## Solvers

The solver name usually follows the convention of `provider-id.backend-name`.

Gurobi has the following solvers:
- `gurobi.qubo` - For Quadratic Unconstrained Binary Optimization (QUBO) problems
- `gurobi.mps` - For Mathematical Programming System (MPS) format problems (supports LP, MIP, and MIQCP including non-convex MIQCP)
- `gurobi.lp` - For Linear Programming (LP) format problems
- `gurobi.presolve` - For presolving MPS problems separately before solving

## Options

Gurobi provides extensive parameter control through `GurobiParameterModel`. Some commonly used parameters include:

**Termination Parameters:**
- `TimeLimit`: Maximum time (in seconds) the solver should run
- `MIPGap`: Relative gap between lower and upper bounds for terminating MIP optimization
- `NodeLimit`: Maximum number of MIP nodes to explore
- `SolutionLimit`: Stop after finding a specified number of feasible solutions

**Tolerances:**
- `FeasibilityTol`: Tolerance for primal feasibility (default: 1e-6)
- `OptimalityTol`: Tolerance for dual feasibility (default: 1e-6)
- `MIPGap`: Relative gap for MIP termination (default: 1e-4)

**Presolve Parameters:**
- `Presolve`: Sets the presolve level (default: -1, meaning automatic)
- `Aggregate`: Controls aggregation level in presolve

For a complete list of parameters, see the [Gurobi Parameters Reference](https://docs.gurobi.com/optimizer/parameters).

In [ ]:
solver = "gurobi.qubo"

from strangeworks_optimization_models.parameter_models import GurobiParameterModel

options = GurobiParameterModel(TimeLimit=60)

## Running a job

In [ ]:
tags = ["foo", "bar"]
so = StrangeworksOptimizer(model=model, solver=solver, options=options, tags=tags)
sw_job = so.run()
print(f"Job slug: {sw_job.slug}")

## Status

In [7]:
print(f"Job status: {so.status(sw_job.slug)}")

Job status: CREATED


## Results

In [8]:
results = so.results(sw_job.slug)
print(f"Best solution:\n{results.solution.first}")

Best solution:
Sample(sample={1: 0, 2: 1, 3: 1, 4: 0, 5: 1}, energy=-5.0, num_occurrences=1)


## Additional Examples

### MPS Format

Gurobi also supports MPS (Mathematical Programming System) format, which is a standard format for linear and mixed-integer programming problems:

**Important Note:** Gurobi supports MIQCP (Mixed-Integer Quadratically Constrained Programming) problems in MPS format, including non-convex MIQCP problems. Gurobi 12.0 introduced significant improvements for solving non-convex MIQCP problems. For more information, see the [Gurobi MIQCP documentation](https://www.gurobi.com/faqs/gurobi-miqcp/).

```python
from strangeworks_optimization import StrangeworksOptimizer
from strangeworks_optimization_models.problem_models import MPSFile
from strangeworks_optimization_models.parameter_models import GurobiParameterModel

# Read an MPS file
# model = MPSFile.read_file("example.mps")

# solver = "gurobi.mps"
# options = GurobiParameterModel(TimeLimit=60)

# so = StrangeworksOptimizer(model=model, solver=solver, options=options)
# sw_job = so.run()
# results = so.results(sw_job.slug)
```

### LP Format

Gurobi also supports LP (Linear Programming) format:

```python
from strangeworks_optimization import StrangeworksOptimizer
from strangeworks_optimization_models.problem_models import LpFile
from strangeworks_optimization_models.parameter_models import GurobiParameterModel

# Read an LP file
# model = LpFile.read_file("example.lp")
# options = GurobiParameterModel(TimeLimit=60)

# so = StrangeworksOptimizer(model=model, solver="gurobi.lp", options=options)
# sw_job = so.run()
# results = so.results(sw_job.slug)
```
